# SAMVAAD — Disfluency Classifier (M7)

Trains the model that detects **blocks, prolongations, repetitions and interjections** in speech.

---

## What this model is for

> **A detected disfluency produces a COACHING CUE. It never produces a score deduction.**

P5 (Karthik) has a stammer. If his stammer lowered his score, SAMVAAD would be telling him daily
that he is failing at not being disabled — the exact harm the product exists to prevent.
Detected events feed a speech-therapy coaching library and the *fluency* dimension of the
Personal Progress Index, which is measured against the learner's **own** rolling baseline.

See `docs/ETHICS_CHARTER.md` rule E1 and `docs/ADR/0003-baseline-relative-scoring.md`.

---

## Before you start

1. **Runtime → Change runtime type → T4 GPU** (free tier is fine).
2. Have `samvaad-training.zip` ready to upload (it is tiny, ~30 KB).
3. Budget **2–4 hours** for the full dataset, or **~25 minutes** for the quick subset run.

Run the cells **in order, top to bottom**. Each one prints what it did.

---

## Target

| Metric | Bar |
|---|---|
| Macro-F1 across the 5 event types | **≥ 0.65** |
| Per-class recall | No class at 0 — the model must not collapse to "no disfluency" |
| Split | **Show-disjoint** — random splits leak speakers and inflate F1 badly |

## Cell 1 — Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — CPU is fine for the LightGBM baseline'

import sys
print('python', sys.version.split()[0])

## Cell 2 — Install dependencies

Takes ~2 minutes. Ignore pip's dependency-resolver warnings.

In [ ]:
%pip install -q librosa soundfile lightgbm scikit-learn onnx onnxruntime skl2onnx pandas tqdm
print('done')

## Cell 3 — Mount Google Drive

Used to save the trained weights so a Colab disconnect does not lose your run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = '/content/drive/MyDrive/samvaad/disfluency'
os.makedirs(OUT, exist_ok=True)
print('saving results to', OUT)

## Cell 4 — Upload `samvaad-training.zip`

A file picker appears. Choose the zip.

**Why this matters:** it contains `pipeline/disfluency.py`, the *same* feature-extraction
function the production service uses. Training on different features from the ones served is
one of the most common and most invisible ML bugs — the model scores well here and behaves
randomly in production, and nothing errors.

In [ ]:
from google.colab import files
import zipfile, sys, os

os.makedirs('/content/samvaad', exist_ok=True)
uploaded = files.upload()

name = next(iter(uploaded))
with zipfile.ZipFile(name) as archive:
    archive.extractall('/content/samvaad')

sys.path.insert(0, '/content/samvaad')

from pipeline.disfluency import (
    LABELS, N_FEATURES, FEATURE_NAMES, SAMPLE_RATE, WINDOW_SECONDS, extract_features,
)

print('labels    :', LABELS)
print('features  :', N_FEATURES)
print('window    :', WINDOW_SECONDS, 's at', SAMPLE_RATE, 'Hz')

## Cell 5 — Get the SEP-28k labels

[SEP-28k](https://github.com/apple/ml-stuttering-events-dataset) is Apple's *Stuttering Events
in Podcasts* dataset: ~28,000 three-second clips, each labelled by three annotators.

The repo ships the **labels and episode URLs**, not the audio — you download the podcasts
yourself in Cell 7.

In [ ]:
!git clone -q https://github.com/apple/ml-stuttering-events-dataset.git /content/sep28k

import pandas as pd

# EpId MUST be read as a string. It is stored with leading whitespace, and
# Apple's own extract_clips.py reads it as str then strips it.
labels = pd.read_csv('/content/sep28k/SEP-28k_labels.csv', dtype={'EpId': str})
labels['EpId'] = labels.EpId.str.strip()
labels['Show'] = labels.Show.str.strip()

# The episodes file has FIVE columns and no header, and the join keys are the
# LAST TWO -- see download_audio.py in the repo, which uses table[i,-2] and
# table[i,-1]. Columns 0 and 1 are the full show title and a URL slug; joining
# on those matches nothing at all.
episodes = pd.read_csv('/content/sep28k/SEP-28k_episodes.csv', header=None, dtype=str)
episodes = episodes.apply(lambda column: column.str.strip())
episodes = episodes.rename(columns={2: 'URL',
                                    episodes.columns[-2]: 'Show',
                                    episodes.columns[-1]: 'EpId'})

overlap = set(zip(episodes.Show, episodes.EpId)) & set(zip(labels.Show, labels.EpId))
print(f'{len(labels):,} clips across {labels.Show.nunique()} shows')
print(f'{len(episodes):,} episodes, {len(overlap):,} of them labelled')

# Fail here rather than silently downloading nothing in Cell 7.
assert overlap, 'episode/label join is broken -- check the CSV column layout'
labels.head(3)

## Cell 6 — Choose your run size

**Do a `quick` run first.** It proves the whole pipeline works end to end in ~25 minutes.
Only then spend hours on the full run.

| Mode | Episodes | Download | Total time |
|---|---|---|---|
| `quick` | 40 | ~1.5 GB | ~25 min |
| `full` | all | ~25 GB | 2–4 h |

In [ ]:
MODE = 'quick'   # <-- change to 'full' for the real run

MAX_EPISODES = 40 if MODE == 'quick' else None

# Only consider episodes we can actually download.
labelled = (labels[['Show', 'EpId']].drop_duplicates()
            .merge(episodes[['Show', 'EpId', 'URL']], on=['Show', 'EpId'], how='inner'))

if MAX_EPISODES:
    # Spread evenly across ALL shows, never the first N rows: the file is
    # ordered by show, so taking the head would train on two speakers and prove
    # nothing. groups=False keeps the grouping columns out of the result.
    per_show = max(1, MAX_EPISODES // labelled.Show.nunique())
    labelled = labelled.groupby('Show', group_keys=False)[labelled.columns].head(per_show)

wanted = set(zip(labelled.Show, labelled.EpId))
print(f'{len(wanted)} episodes from {labelled.Show.nunique()} shows')
assert wanted, 'no downloadable episodes selected'

## Cell 7 — Download the podcast audio

**The long one.** It is resumable: if Colab disconnects, just re-run this cell and it skips
what it already has.

Some URLs will 404 — podcast feeds rot. That is expected and harmless; the cell reports how
many succeeded.

In [ ]:
import os, requests
from tqdm.auto import tqdm

AUDIO = '/content/audio'
os.makedirs(AUDIO, exist_ok=True)

todo = labelled.reset_index(drop=True)
print(f'{len(todo)} episodes to fetch')

ok = failed = skipped = 0

for row in tqdm(list(todo.itertuples()), desc='episodes'):
    path = f'{AUDIO}/{row.Show}_{row.EpId}.mp3'
    if os.path.exists(path) and os.path.getsize(path) > 10_000:
        skipped += 1
        continue
    try:
        with requests.get(row.URL, stream=True, timeout=120) as response:
            response.raise_for_status()
            with open(path, 'wb') as handle:
                for chunk in response.iter_content(1 << 20):
                    handle.write(chunk)
        ok += 1
    except Exception:
        failed += 1
        if os.path.exists(path):
            os.remove(path)

print(f'downloaded {ok}, already had {skipped}, unavailable {failed}')
assert ok + skipped > 0, 'nothing downloaded -- check the URLs in `labelled`'

## Cell 8 — Build the labelled clip table

Each clip was rated by 3 annotators, so each label is a count 0–3. We binarise at
**≥ 2 of 3 agreeing**, the convention used in the SEP-28k paper.

Clips marked *unsure*, *poor audio*, *music* or *no speech* are dropped: training on audio the
annotators themselves could not judge just teaches the model to be confidently wrong.

In [ ]:
AGREEMENT = 2

SOURCE_COLUMNS = {
    'block': 'Block',
    'prolongation': 'Prolongation',
    'sound_repetition': 'SoundRep',
    'word_repetition': 'WordRep',
    'interjection': 'Interjection',
}

# Which episodes actually made it to disk. EpId stays a STRING throughout -
# comparing '0' to 0 silently matches nothing, which is what emptied Cell 7.
downloaded = set()
for filename in os.listdir(AUDIO):
    if filename.endswith('.mp3'):
        show, ep_id = filename[:-4].rsplit('_', 1)
        downloaded.add((show, ep_id))

clips = labels[[(s, e) in downloaded for s, e in zip(labels.Show, labels.EpId)]].copy()
print(f'{len(clips):,} clips from {len(downloaded)} downloaded episodes')
assert len(clips), 'no labelled clips matched the downloaded audio'

# Drop audio the annotators themselves could not judge. Training on it just
# teaches the model to be confidently wrong.
unusable = ((clips.Unsure >= AGREEMENT) | (clips.PoorAudioQuality >= AGREEMENT)
            | (clips.Music >= AGREEMENT) | (clips.NoSpeech >= AGREEMENT))
clips = clips[~unusable]

for label, column in SOURCE_COLUMNS.items():
    clips[label] = (clips[column] >= AGREEMENT).astype(int)

print(f'{len(clips):,} usable clips\n')
print('positives per class:')
for label in LABELS:
    n = int(clips[label].sum())
    print(f'  {label:18} {n:6,}  ({n / len(clips):5.1%})')
print(f'\n  {"no event":18} {int((clips[LABELS].sum(axis=1) == 0).sum()):6,}')

## Cell 9 — Extract features

Uses `extract_features` **imported from the zip** — the identical function the service runs at
inference time.

This is the CPU-heavy cell. Roughly 8 minutes for the quick run.

In [ ]:
import numpy as np, librosa, warnings
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

X, Y, groups = [], [], []
cache = {}

for row in tqdm(list(clips.itertuples()), desc='clips'):
    key = (row.Show, row.EpId)
    if key not in cache:
        cache.clear()  # one episode in memory at a time
        try:
            cache[key] = librosa.load(f'{AUDIO}/{row.Show}_{row.EpId}.mp3', sr=SAMPLE_RATE)[0]
        except Exception:
            cache[key] = None

    audio = cache[key]
    if audio is None:
        continue

    start, stop = int(row.Start), int(row.Stop)
    segment = audio[start:stop]
    if len(segment) < SAMPLE_RATE // 2:
        continue

    X.append(extract_features(segment))
    Y.append([getattr(row, label) for label in LABELS])
    groups.append(row.Show)   # show-disjoint splitting happens on this

X = np.vstack(X).astype(np.float32)
Y = np.array(Y, dtype=np.int8)
groups = np.array(groups)

print('X', X.shape, ' Y', Y.shape)
assert X.shape[1] == N_FEATURES, 'feature count drifted from the service'
np.savez_compressed(f'{OUT}/features.npz', X=X, Y=Y, groups=groups)
print('cached to Drive')

## Cell 10 — Split by show, never at random

Clips from one podcast share a speaker, a microphone and a room. A random split puts the same
speaker in train and test, and the model scores brilliantly by recognising the voice rather
than the disfluency. Published stuttering-detection results differ by **20+ F1 points** on this
choice alone.

A show-disjoint split is the honest one, and the number it gives you is the one that will hold
up on a real learner.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

outer = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_val_idx, test_idx = next(outer.split(X, Y, groups))

inner = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_rel, val_rel = next(inner.split(X[train_val_idx], Y[train_val_idx], groups[train_val_idx]))
train_idx, val_idx = train_val_idx[train_rel], train_val_idx[val_rel]

for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    print(f'{name:6} {len(idx):6,} clips  {len(set(groups[idx])):3} shows')

assert not (set(groups[train_idx]) & set(groups[test_idx])), 'shows leaked between splits'
print('\nno show appears in more than one split')

## Cell 11 — Train

One binary LightGBM per event type (multi-label: a clip can be both a block and a
prolongation). `scale_pos_weight` handles the heavy class imbalance — without it the model
learns to predict "no disfluency" for everything and posts a great accuracy while being
completely useless.

In [ ]:
import lightgbm as lgb

models = {}

for i, label in enumerate(LABELS):
    y_train, y_val = Y[train_idx, i], Y[val_idx, i]
    positives = int(y_train.sum())
    if positives < 20:
        print(f'{label}: only {positives} positives — skipping (use MODE="full")')
        continue

    model = lgb.LGBMClassifier(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8,
        scale_pos_weight=float((len(y_train) - positives) / max(positives, 1)),
        random_state=42, verbose=-1,
    )
    model.fit(X[train_idx], y_train,
              eval_set=[(X[val_idx], y_val)],
              eval_metric='auc',
              callbacks=[lgb.early_stopping(50, verbose=False)])

    models[label] = model
    print(f'{label:18} trained on {positives:,} positives, best iter {model.best_iteration_}')

## Cell 12 — Tune thresholds on validation

0.5 is the wrong threshold for imbalanced classes. Each event type gets the threshold that
maximises its F1 **on validation** — never on test, which would leak.

In [ ]:
from sklearn.metrics import f1_score

thresholds = {}

for i, label in enumerate(LABELS):
    if label not in models:
        continue
    probabilities = models[label].predict_proba(X[val_idx])[:, 1]
    grid = np.arange(0.05, 0.96, 0.01)
    scores = [f1_score(Y[val_idx, i], probabilities >= t, zero_division=0) for t in grid]
    thresholds[label] = float(grid[int(np.argmax(scores))])
    print(f'{label:18} threshold {thresholds[label]:.2f}  val F1 {max(scores):.3f}')

## Cell 13 — Evaluate on the held-out test shows

**This is the number that counts.** The target is macro-F1 ≥ 0.65.

Per-class recall is printed too, and matters as much as the headline: a model that never
predicts *block* is useless to the learner it was built for, however good its average looks.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
import json

rows, f1s = [], []

print(f'{"event":18} {"prec":>6} {"recall":>7} {"F1":>6} {"AUC":>6} {"n":>6}')
print('-' * 54)

for i, label in enumerate(LABELS):
    if label not in models:
        continue
    y_true = Y[test_idx, i]
    probabilities = models[label].predict_proba(X[test_idx])[:, 1]
    y_pred = (probabilities >= thresholds[label]).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0)
    auc = roc_auc_score(y_true, probabilities) if len(set(y_true)) > 1 else float('nan')

    f1s.append(f1)
    rows.append({'event': label, 'precision': precision, 'recall': recall,
                 'f1': f1, 'auc': auc, 'support': int(y_true.sum()),
                 'threshold': thresholds[label]})
    print(f'{label:18} {precision:6.3f} {recall:7.3f} {f1:6.3f} {auc:6.3f} {int(y_true.sum()):6,}')

macro_f1 = float(np.mean(f1s))
print('-' * 54)
print(f'{"MACRO-F1":18} {macro_f1:20.3f}   target >= 0.65')

collapsed = [r['event'] for r in rows if r['recall'] < 0.05]
if collapsed:
    print(f'\nWARNING: near-zero recall for {collapsed} — the model is ignoring these events.')

print('\nPASS' if macro_f1 >= 0.65 and not collapsed else '\nBELOW TARGET — see Cell 16')

## Cell 14 — What the model is actually listening to

A sanity check, not decoration. If *block* is not driven by the silence features, something is
wrong — a block **is** a silent closure, and a model reaching that label some other way has
learned a shortcut that will not survive contact with a real learner.

In [ ]:
for label, model in models.items():
    order = np.argsort(model.feature_importances_)[::-1][:6]
    print(f'{label:18} ' + ', '.join(FEATURE_NAMES[j] for j in order))

## Cell 15 — Export to ONNX and save everything

ONNX so the service runs it on CPU with no PyTorch or LightGBM installed — which is what keeps
the speech container small enough for the free tier.

In [ ]:
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType
import datetime, shutil

sample = X[:1].astype(np.float32)

for label, model in models.items():
    onx = to_onnx(model, sample, target_opset=15,
                  options={id(model): {'zipmap': False}},
                  initial_types=[('features', FloatTensorType([None, N_FEATURES]))])
    with open(f'{OUT}/disfluency_{label}.onnx', 'wb') as handle:
        handle.write(onx.SerializeToString())

metrics = {
    'trained_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'mode': MODE,
    'macro_f1': macro_f1,
    'target_macro_f1': 0.65,
    'passed': bool(macro_f1 >= 0.65 and not collapsed),
    'per_class': rows,
    'thresholds': thresholds,
    'n_train': int(len(train_idx)), 'n_val': int(len(val_idx)), 'n_test': int(len(test_idx)),
    'train_shows': sorted(set(groups[train_idx])),
    'test_shows': sorted(set(groups[test_idx])),
    'feature_names': FEATURE_NAMES,
    'n_features': N_FEATURES,
    'split': 'show-disjoint GroupShuffleSplit(seed=42)',
    'label_agreement': AGREEMENT,
}

with open(f'{OUT}/metrics.json', 'w') as handle:
    json.dump(metrics, handle, indent=2)

shutil.make_archive('/content/disfluency_model', 'zip', OUT)
print('saved to', OUT)
print('\nfiles:', sorted(os.listdir(OUT)))

In [ ]:
from google.colab import files
files.download('/content/disfluency_model.zip')

## Cell 16 — If macro-F1 is below 0.65

In order of expected payoff:

1. **Set `MODE = 'full'` in Cell 6 and re-run.** The quick subset is usually the whole reason.
   More data helps this task more than any other change.
2. **Check which class is dragging.** `block` is usually the hardest; `interjection` the
   easiest. If one class has near-zero recall it has too few positives — that is a data
   problem, not a model problem.
3. **Try the wav2vec2 features** (Approach B in the plan): replace Cell 9's feature extraction
   with a frozen `facebook/wav2vec2-base` encoder and mean-pool its hidden states. Needs a GPU
   and roughly 40 minutes. Expect +0.05–0.10 macro-F1.
4. **Add FluencyBank** — the same repo ships `fluencybank_labels.csv` in the identical format.

**Do not** tune thresholds on the test set to hit the number. The bar exists to tell us whether
the model is good enough to put in front of a person who stammers, and a number obtained that
way answers a different question.

---

## Cell 17 — Handing back

Send back **`disfluency_model.zip`**. It contains:

```
disfluency_block.onnx
disfluency_prolongation.onnx
disfluency_sound_repetition.onnx
disfluency_word_repetition.onnx
disfluency_interjection.onnx
metrics.json          <- the eval table that goes in the PR description
features.npz          <- optional; lets the split be reproduced without re-downloading
```

Those land in `services/speech/artifacts/` (gitignored — weights belong in a model registry,
not in git), and `metrics.json` becomes the eval table the speech CI job prints.

**Even a failing run is worth sending back.** A model at 0.55 with honest per-class numbers is
more useful than no model, and it tells us exactly which event type needs more data.